In [0]:
SRC = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
DST = "workspace.default"

for t in ["facilities", "india_post_pincode_directory", "nfhs_5_district_health_indicators"]:
    spark.table(f"{SRC}.{t}").write.mode("overwrite").saveAsTable(f"{DST}.bronze_{t}")
    print(t, spark.table(f"{DST}.bronze_{t}").count(), "rows")

facilities 10088 rows
india_post_pincode_directory 165627 rows
nfhs_5_district_health_indicators 706 rows


In [0]:
fac = spark.table("workspace.default.bronze_facilities")
for name, dtype in fac.dtypes:
    print(f"{name:40s} {dtype}")

unique_id                                string
source_types                             string
source_ids                               string
source_content_id                        string
name                                     string
organization_type                        string
content_table_id                         string
phone_numbers                            string
officialPhone                            string
email                                    string
websites                                 string
officialWebsite                          string
yearEstablished                          string
acceptsVolunteers                        string
facebookLink                             string
address_line1                            string
address_line2                            string
address_line3                            string
address_city                             string
address_stateOrRegion                    string
address_zipOrPostcode                   

In [0]:
import pandas as pd
pdf = spark.table("workspace.default.bronze_facilities").toPandas()
EMPTY = {"", "null", "none", "[]", "nan", "{}"}
is_empty = lambda v: v is None or (isinstance(v, float) and pd.isna(v)) or str(v).strip().lower() in EMPTY
print(len(pdf), "rows,", len(pdf.columns), "columns")
for c in pdf.columns: print(f"{c:32s} missing {pdf[c].apply(is_empty).mean()*100:5.1f}%  e.g. {next((str(v) for v in pdf[c] if not is_empty(v)), '')[:100]}")

10088 rows, 51 columns
unique_id                        missing   0.0%  e.g. b8a5401f-42f1-422a-8cd9-686a15b4cb76
source_types                     missing   0.8%  e.g. ["dynamic","overture","constant","mongo_facility","dynamic","overture","constant","mongo_facility","
source_ids                       missing   7.8%  e.g. ["08f42d99986e2b6d03862b16be6f4556","08f3cb3a2a1a139503b31af626a9e452","08f52b1ca500bb5c034e6665048c
source_content_id                missing   0.7%  e.g. c2ada12b-11f7-45f2-93b6-94f7e8853c7a
name                             missing   0.6%  e.g. Aravind Eye Hospital
organization_type                missing   0.6%  e.g. facility
content_table_id                 missing   0.6%  e.g. c2ada12b-11f7-45f2-93b6-94f7e8853c7a
phone_numbers                    missing   3.4%  e.g. ["+911123233400","+911123232400","+911126142832","+911126141119","+911120895995","+919910631876","+9
officialPhone                    missing   5.6%  e.g. +911123233400
email                            

In [0]:
import json
cols = ["name","address_city","address_stateOrRegion","address_zipOrPostcode","facilityTypeId","operatorTypeId","numberDoctors","capacity","yearEstablished","description","specialties","procedure","equipment","capability","websites"]
samp = pdf[pdf["procedure"].astype(str).str.contains("ialysis", na=False)].sample(2, random_state=3)
for _, r in samp.iterrows(): print("="*90); [print(f"{c}: {str(r[c])[:400]}") for c in cols]
print("\n### facilityTypeId:", pdf["facilityTypeId"].value_counts().head(10).to_dict())
print("### operatorTypeId:", pdf["operatorTypeId"].value_counts().head(5).to_dict())
print("### top states:", pdf["address_stateOrRegion"].value_counts().head(12).to_dict())
lat = pd.to_numeric(pdf["latitude"], errors="coerce"); lon = pd.to_numeric(pdf["longitude"], errors="coerce")
print("### coords outside India box:", int(((lat<6)|(lat>37)|(lon<68)|(lon>98)).sum()))
print("### duplicate name+city:", int(pdf.duplicated(subset=["name","address_city"]).sum()))
spec = pdf["specialties"].dropna().apply(lambda s: list(set(json.loads(s))) if str(s).startswith("[") else [])
print("### distinct specialties:", spec.explode().nunique(), "| top:", spec.explode().value_counts().head(25).to_dict())
for kw in ["dialysis","nicu","neonatal","caesarean|c-section|cesarean","mri","burn","cardiac surgery|cardiothoracic","trauma","icu"]: print(f"### mentions '{kw}':", int(pdf[["description","procedure","equipment","capability","specialties"]].astype(str).apply(lambda col: col.str.contains(kw, case=False, regex=True)).any(axis=1).sum()))

name: Dr Hedgewar Hospital
address_city: Aurangabad
address_stateOrRegion: Maharashtra
address_zipOrPostcode: 431005
facilityTypeId: hospital
operatorTypeId: private
numberDoctors: 58
capacity: 500
yearEstablished: 2011
description: Listed as Ward 7 hospital in Ward 7, Jawahar Colony, F-7 Jawahar Colony, Chhatrapati Sambhajinagar, Maharashtra, India.
specialties: ["shoulderAndElbowOrthopedicSurgery","orthopedicSurgery","internalMedicine","internalMedicine","internalMedicine","internalMedicine","internalMedicine","dentistry","generalSurgery","internalMedicine","internalMedicine","cardiology","otolaryngology","neurosurgery","urology","gynecologyAndObstetrics","orthopedicSurgery","ophthalmology","plasticSurgery","gastroenterology","nephrology","surgicalOncolo
procedure: ["Laparoscopic Hysterectomy","Arthroscopy","Total Hip Replacement","Total Knee Replacement","Piles Laser Treatment","Inguinal Hernia Surgery","Laparoscopic Cholecystectomy","Laser Surgery for Fissure","Laser Surgery for Fi

In [0]:
%sql
GRANT ALL PRIVILEGES ON SCHEMA workspace.default TO `timwang007@outlook.com`;
GRANT ALL PRIVILEGES ON SCHEMA workspace.default TO `<user>`;